# Práctica Final UT4: Limpieza, Transformación y Análisis de Datos con Apache Spark

**Alumno:** Abel Gijón  
**Dataset elegido:** Jena Climate Dataset  
**Entorno de trabajo:** Databricks con PySpark  
**Formato de entrada:** CSV   

En este notebook se desarrolla un flujo completo de tratamiento de datos con Spark DataFrames.  
El objetivo es cargar un dataset climático real, inspeccionar su calidad, limpiar registros incoherentes, crear nuevas variables útiles para el análisis y obtener conclusiones mediante consultas con DataFrames y Spark SQL.

## Configuración inicial

El dataset utilizado contiene mediciones meteorológicas tomadas cada 10 minutos.  
Incluye variables como temperatura, humedad, presión atmosférica, velocidad del viento y dirección del viento.

La salida final se guardará en formato **Parquet**, ya que es un formato columnar eficiente, comprimido y perfecto para entornos Big Data.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window

# Ruta del dataset en Databricks.
DATA_PATH = "/Volumes/workspace/default/dataset/jena_climate_2009_2016.csv"

# Ruta de salida del dataset limpio.
OUTPUT_PATH_PARQUET = "/Volumes/workspace/default/dataset/jena_climate_limpio.parquet"

print("Ruta de entrada:", DATA_PATH)
print("Ruta de salida:", OUTPUT_PATH_PARQUET)

Ruta de entrada: /Volumes/workspace/default/dataset/jena_climate_2009_2016.csv
Ruta de salida: /Volumes/workspace/default/dataset/jena_climate_limpio.parquet


## Carga de datos

Se carga el CSV con Spark, indicando que la primera fila contiene cabeceras.  
También se activa la inferencia de esquema para que Spark intente detectar automáticamente los tipos de datos.

En pasos posteriores corregiremos los tipos de forma explícita, especialmente la columna de fecha.

In [0]:
# Cargar el dataset original.
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "false")
    .option("escape", '"')
    .csv(DATA_PATH)
)

# Mostrar el número de registros y columnas cargados.
print("Registros cargados:", df_raw.count())
print("Columnas cargadas:", len(df_raw.columns))

Registros cargados: 420551
Columnas cargadas: 15


## Revisión de la estructura inicial del DataFrame:

- número total de filas;
- número total de columnas;
- nombres de columnas;
- tipos de datos detectados por Spark;
- primeras filas del dataset.


In [0]:
# Obtener el número total de filas y columnas.
total_filas = df_raw.count()
total_columnas = len(df_raw.columns)

# Imprimir la información básica del dataset.
print("Número total de filas:", total_filas)
print("Número total de columnas:", total_columnas)

print("\nColumnas del dataset:")
for c in df_raw.columns:
    print("-", c)

# Mostrar los tipos de datos detectados por Spark.
print("\nTipos de datos detectados por Spark:")
df_raw.printSchema()

# Mostrar las primeras filas del dataset.
display(df_raw.limit(10))

Número total de filas: 420551
Número total de columnas: 15

Columnas del dataset:
- Date Time
- p (mbar)
- T (degC)
- Tpot (K)
- Tdew (degC)
- rh (%)
- VPmax (mbar)
- VPact (mbar)
- VPdef (mbar)
- sh (g/kg)
- H2OC (mmol/mol)
- rho (g/m**3)
- wv (m/s)
- max. wv (m/s)
- wd (deg)

Tipos de datos detectados por Spark:
root
 |-- Date Time: string (nullable = true)
 |-- p (mbar): double (nullable = true)
 |-- T (degC): double (nullable = true)
 |-- Tpot (K): double (nullable = true)
 |-- Tdew (degC): double (nullable = true)
 |-- rh (%): double (nullable = true)
 |-- VPmax (mbar): double (nullable = true)
 |-- VPact (mbar): double (nullable = true)
 |-- VPdef (mbar): double (nullable = true)
 |-- sh (g/kg): double (nullable = true)
 |-- H2OC (mmol/mol): double (nullable = true)
 |-- rho (g/m**3): double (nullable = true)
 |-- wv (m/s): double (nullable = true)
 |-- max. wv (m/s): double (nullable = true)
 |-- wd (deg): double (nullable = true)



Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
01.01.2009 00:10:00,996.52,-8.02,265.4,-8.9,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.8,0.72,1.5,136.1
01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.2,1.88,3.02,1310.24,0.19,0.63,171.6
01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.5,198.0
01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.0,0.32,0.63,214.3
01.01.2009 01:00:00,996.5,-8.05,265.38,-8.78,94.4,3.33,3.14,0.19,1.96,3.15,1307.86,0.21,0.63,192.7
01.01.2009 01:10:00,996.5,-7.62,265.81,-8.3,94.8,3.44,3.26,0.18,2.04,3.27,1305.68,0.18,0.63,166.5
01.01.2009 01:20:00,996.5,-7.62,265.81,-8.36,94.4,3.44,3.25,0.19,2.03,3.26,1305.69,0.19,0.5,118.6
01.01.2009 01:30:00,996.5,-7.91,265.52,-8.73,93.8,3.36,3.15,0.21,1.97,3.16,1307.17,0.28,0.75,188.5
01.01.2009 01:40:00,996.53,-8.43,264.99,-9.34,93.1,3.23,3.0,0.22,1.88,3.02,1309.85,0.59,0.88,185.0


## Análisis inicial de calidad

Se revisan los principales problemas de calidad:

- valores nulos por columna;
- registros duplicados completos;
- fechas duplicadas;
- tipos de datos incorrectos;
- posibles valores incoherentes en variables meteorológicas.

En datasets de sensores es habitual revisar rangos físicos, por ejemplo humedad relativa entre 0 y 100, velocidades de viento no negativas y dirección del viento entre 0 y 360 grados.

In [0]:
# Valores nulos por columna.

df_nulos = df_raw.select([
    # F.expr con backticks permite referenciar correctamente columnas con nombres complejos.... max. wv (m/s), wv (m/s), wd (deg)
    F.expr(f"sum(case when `{c}` is null then 1 else 0 end) as `{c}`")
    for c in df_raw.columns
])

display(df_nulos)

Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Duplicados completos
duplicados_completos = df_raw.count() - df_raw.dropDuplicates().count()
print("Registros duplicados completos:", duplicados_completos)

# Fechas duplicadas en la columna original Date Time
df_fechas_duplicadas = (
    df_raw
    .groupBy("Date Time")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Número de timestamps duplicados:", df_fechas_duplicadas.count())
display(df_fechas_duplicadas.limit(20))

Registros duplicados completos: 327
Número de timestamps duplicados: 327


Date Time,count
01.07.2010 06:50:00,2
01.07.2010 11:20:00,2
01.07.2010 11:10:00,2
01.07.2010 23:20:00,2
01.07.2010 04:20:00,2
01.07.2010 13:20:00,2
01.07.2010 05:40:00,2
01.07.2010 04:30:00,2
01.07.2010 14:00:00,2
01.07.2010 00:10:00,2


In [0]:
# Resumen estadístico inicial de columnas numéricas.
# Creamos alias temporales limpios para evitar problemas con espacios, puntos, paréntesis y símbolos.

columnas_numericas_raw = {
    "p (mbar)": "presion_mbar",
    "T (degC)": "temperatura_c",
    "Tpot (K)": "temperatura_potencial_k",
    "Tdew (degC)": "temperatura_rocio_c",
    "rh (%)": "humedad_relativa",
    "VPmax (mbar)": "presion_vapor_max_mbar",
    "VPact (mbar)": "presion_vapor_actual_mbar",
    "VPdef (mbar)": "deficit_presion_vapor_mbar",
    "sh (g/kg)": "humedad_especifica_gkg",
    "H2OC (mmol/mol)": "concentracion_vapor_mmolmol",
    "rho (g/m**3)": "densidad_aire_gm3",
    "wv (m/s)": "velocidad_viento_ms",
    "max. wv (m/s)": "velocidad_viento_max_ms",
    "wd (deg)": "direccion_viento_grados"
}

df_numerico_raw = df_raw.select([
    F.col(f"`{col_original}`").alias(col_limpia)
    for col_original, col_limpia in columnas_numericas_raw.items()
])

display(df_numerico_raw.describe())

summary,presion_mbar,temperatura_c,temperatura_potencial_k,temperatura_rocio_c,humedad_relativa,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_gkg,concentracion_vapor_mmolmol,densidad_aire_gm3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados
count,420551,420551,420551,420551,420551,420551,420551,420551,420551,420551,420551,420551,420551,420551
mean,989.2127760961208,9.45014735430429,283.49274344847663,4.955853844123552,76.00825940254575,13.576250537984718,9.53375590594245,4.042411574339375,6.0224082929299705,9.640223112060127,1216.0627478950232,1.702223844432657,3.056555257269718,174.74373838131402
stddev,8.358480696599688,8.423365210385157,8.50447139145782,6.730674307693392,16.47617537158425,7.73902005741365,4.184164339702183,4.896850891024253,2.6561390257685615,4.235394815093718,39.97520827984654,65.44671380682638,69.01693185330184,86.6816927461434
min,913.6,-23.01,250.6,-25.01,12.95,0.95,0.79,0.0,0.5,0.8,1059.45,-9999.0,-9999.0,0.0
max,1015.35,37.28,311.34,23.11,100.0,63.77,28.32,46.01,18.13,28.82,1393.54,28.49,23.5,360.0


## Renombrado de columnas

Las columnas originales contienen espacios, paréntesis, símbolos y unidades.  
Para trabajar de forma más cómoda, se renombran usando nombres en minúsculas, sin espacios ni caracteres especiales.

También se mantienen nombres descriptivos para que el notebook sea más legible.

In [0]:
# Listado de columnas originales y sus nuevos nombres limpios.
columnas_renombradas = {
    "Date Time": "fecha_hora",
    "p (mbar)": "presion_mbar",
    "T (degC)": "temperatura_c",
    "Tpot (K)": "temperatura_potencial_k",
    "Tdew (degC)": "punto_rocio_c",
    "rh (%)": "humedad_relativa_pct",
    "VPmax (mbar)": "presion_vapor_max_mbar",
    "VPact (mbar)": "presion_vapor_actual_mbar",
    "VPdef (mbar)": "deficit_presion_vapor_mbar",
    "sh (g/kg)": "humedad_especifica_g_kg",
    "H2OC (mmol/mol)": "concentracion_agua_mmol_mol",
    "rho (g/m**3)": "densidad_aire_g_m3",
    "wv (m/s)": "velocidad_viento_ms",
    "max. wv (m/s)": "velocidad_viento_max_ms",
    "wd (deg)": "direccion_viento_grados"
}

# Aplicar el renombrado de columnas
df_renombrado = df_raw
for origen, destino in columnas_renombradas.items():
    df_renombrado = df_renombrado.withColumnRenamed(origen, destino)

# Mostrar el nuevo esquema y las primeras filas para verificar el renombrado.
df_renombrado.printSchema()
display(df_renombrado.limit(5))

root
 |-- fecha_hora: string (nullable = true)
 |-- presion_mbar: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- temperatura_potencial_k: double (nullable = true)
 |-- punto_rocio_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- presion_vapor_max_mbar: double (nullable = true)
 |-- presion_vapor_actual_mbar: double (nullable = true)
 |-- deficit_presion_vapor_mbar: double (nullable = true)
 |-- humedad_especifica_g_kg: double (nullable = true)
 |-- concentracion_agua_mmol_mol: double (nullable = true)
 |-- densidad_aire_g_m3: double (nullable = true)
 |-- velocidad_viento_ms: double (nullable = true)
 |-- velocidad_viento_max_ms: double (nullable = true)
 |-- direccion_viento_grados: double (nullable = true)



fecha_hora,presion_mbar,temperatura_c,temperatura_potencial_k,punto_rocio_c,humedad_relativa_pct,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_g_kg,concentracion_agua_mmol_mol,densidad_aire_g_m3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados
01.01.2009 00:10:00,996.52,-8.02,265.4,-8.9,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.8,0.72,1.5,136.1
01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.2,1.88,3.02,1310.24,0.19,0.63,171.6
01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.5,198.0
01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.0,0.32,0.63,214.3


## Corrección de tipos de datos

La fecha se convierte a tipo `timestamp` usando el formato original del CSV: `dd.MM.yyyy HH:mm:ss`.

Las variables meteorológicas se convierten explícitamente a tipo numérico `double`.  
De esta forma evitamos errores en agregaciones, filtros y cálculos posteriores.

In [0]:
# Listado de columnas numéricas para tipado posterior.
columnas_numericas = [
    "presion_mbar",
    "temperatura_c",
    "temperatura_potencial_k",
    "punto_rocio_c",
    "humedad_relativa_pct",
    "presion_vapor_max_mbar",
    "presion_vapor_actual_mbar",
    "deficit_presion_vapor_mbar",
    "humedad_especifica_g_kg",
    "concentracion_agua_mmol_mol",
    "densidad_aire_g_m3",
    "velocidad_viento_ms",
    "velocidad_viento_max_ms",
    "direccion_viento_grados"
]

# Tipar columnas numéricas y convertir fecha_hora a timestamp.
df_tipado = df_renombrado.withColumn(
    "fecha_hora",
    F.to_timestamp("fecha_hora", "dd.MM.yyyy HH:mm:ss")
)

for c in columnas_numericas:
    df_tipado = df_tipado.withColumn(c, F.col(c).cast(DoubleType()))

# Mostrar el esquema final y las primeras filas para verificar el tipado.
df_tipado.printSchema()
display(df_tipado.limit(5))

root
 |-- fecha_hora: timestamp (nullable = true)
 |-- presion_mbar: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- temperatura_potencial_k: double (nullable = true)
 |-- punto_rocio_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- presion_vapor_max_mbar: double (nullable = true)
 |-- presion_vapor_actual_mbar: double (nullable = true)
 |-- deficit_presion_vapor_mbar: double (nullable = true)
 |-- humedad_especifica_g_kg: double (nullable = true)
 |-- concentracion_agua_mmol_mol: double (nullable = true)
 |-- densidad_aire_g_m3: double (nullable = true)
 |-- velocidad_viento_ms: double (nullable = true)
 |-- velocidad_viento_max_ms: double (nullable = true)
 |-- direccion_viento_grados: double (nullable = true)



fecha_hora,presion_mbar,temperatura_c,temperatura_potencial_k,punto_rocio_c,humedad_relativa_pct,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_g_kg,concentracion_agua_mmol_mol,densidad_aire_g_m3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados
2009-01-01T00:10:00.000Z,996.52,-8.02,265.4,-8.9,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
2009-01-01T00:20:00.000Z,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.8,0.72,1.5,136.1
2009-01-01T00:30:00.000Z,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.2,1.88,3.02,1310.24,0.19,0.63,171.6
2009-01-01T00:40:00.000Z,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.5,198.0
2009-01-01T00:50:00.000Z,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.0,0.32,0.63,214.3


## Detección de valores incoherentes o atípicos

Antes de eliminar registros, se crean indicadores para detectar posibles problemas:

- humedad relativa fuera del rango 0-100;
- velocidad de viento negativa;
- velocidad máxima de viento inferior a la velocidad media;
- dirección del viento fuera del rango 0-360;
- temperaturas extremadamente bajas o altas;
- presión atmosférica fuera de un rango razonable;
- fechas nulas.

Estos criterios se aplican como reglas de calidad.

In [0]:
# Evaluación de calidad de datos con reglas definidas.
df_calidad = (
    df_tipado
    .withColumn("error_fecha", F.col("fecha_hora").isNull())
    .withColumn("error_humedad", (F.col("humedad_relativa_pct") < 0) | (F.col("humedad_relativa_pct") > 100))
    .withColumn("error_viento_negativo", (F.col("velocidad_viento_ms") < 0) | (F.col("velocidad_viento_max_ms") < 0))
    .withColumn("error_viento_max", F.col("velocidad_viento_max_ms") < F.col("velocidad_viento_ms"))
    .withColumn("error_direccion_viento", (F.col("direccion_viento_grados") < 0) | (F.col("direccion_viento_grados") > 360))
    .withColumn("error_temperatura_extrema", (F.col("temperatura_c") < -60) | (F.col("temperatura_c") > 60))
    .withColumn("error_presion", (F.col("presion_mbar") < 850) | (F.col("presion_mbar") > 1100))
)

# Resumen de errores por tipo.
columnas_error = [
    "error_fecha",
    "error_humedad",
    "error_viento_negativo",
    "error_viento_max",
    "error_direccion_viento",
    "error_temperatura_extrema",
    "error_presion"
]

# Contar el número de errores por tipo.
resumen_errores = df_calidad.select([
    F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c)
    for c in columnas_error
])

# Mostrar el resumen de errores.
display(resumen_errores)

error_fecha,error_humedad,error_viento_negativo,error_viento_max,error_direccion_viento,error_temperatura_extrema,error_presion
0,0,20,2,0,0,0


## Limpieza de datos

En esta fase se aplica la limpieza principal:

1. eliminación de duplicados completos;
2. eliminación de registros sin fecha válida;
3. eliminación de registros con nulos en columnas meteorológicas principales;
4. filtrado de valores físicamente incoherentes;
5. ordenación cronológica del dataset.

En este caso no se eliminan columnas relevantes porque todas las variables originales aportan información útil para el análisis climático.

In [0]:
# Aplicar limpieza de datos eliminando registros con errores y filas duplicadas.

# Contar filas antes de limpieza para comparar posteriormente.
filas_antes_limpieza = df_calidad.count()

# Aplicar las reglas de limpieza: eliminar filas con errores y duplicados, y ordenar por fecha.
df_limpio = (
    df_calidad
    .dropDuplicates()
    .filter(F.col("fecha_hora").isNotNull())
    .dropna(subset=columnas_numericas)
    .filter((F.col("humedad_relativa_pct") >= 0) & (F.col("humedad_relativa_pct") <= 100))
    .filter((F.col("velocidad_viento_ms") >= 0) & (F.col("velocidad_viento_max_ms") >= 0))
    .filter(F.col("velocidad_viento_max_ms") >= F.col("velocidad_viento_ms"))
    .filter((F.col("direccion_viento_grados") >= 0) & (F.col("direccion_viento_grados") <= 360))
    .filter((F.col("temperatura_c") >= -60) & (F.col("temperatura_c") <= 60))
    .filter((F.col("presion_mbar") >= 850) & (F.col("presion_mbar") <= 1100))
    .drop(*columnas_error)
    .orderBy("fecha_hora")
)

# Contar filas después de limpieza para comparar con el conteo inicial.
filas_despues_limpieza = df_limpio.count()

# Imprimir resultados de limpieza.
print("Filas antes de limpieza:", filas_antes_limpieza)
print("Filas después de limpieza:", filas_despues_limpieza)
print("Filas eliminadas:", filas_antes_limpieza - filas_despues_limpieza)

# Mostrar las primeras filas del DataFrame limpio para verificar el resultado.
display(df_limpio.limit(10))

Filas antes de limpieza: 420551
Filas después de limpieza: 420204
Filas eliminadas: 347


fecha_hora,presion_mbar,temperatura_c,temperatura_potencial_k,punto_rocio_c,humedad_relativa_pct,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_g_kg,concentracion_agua_mmol_mol,densidad_aire_g_m3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados
2009-01-01T00:10:00.000Z,996.52,-8.02,265.4,-8.9,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
2009-01-01T00:20:00.000Z,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.8,0.72,1.5,136.1
2009-01-01T00:30:00.000Z,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.2,1.88,3.02,1310.24,0.19,0.63,171.6
2009-01-01T00:40:00.000Z,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.5,198.0
2009-01-01T00:50:00.000Z,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.0,0.32,0.63,214.3
2009-01-01T01:00:00.000Z,996.5,-8.05,265.38,-8.78,94.4,3.33,3.14,0.19,1.96,3.15,1307.86,0.21,0.63,192.7
2009-01-01T01:10:00.000Z,996.5,-7.62,265.81,-8.3,94.8,3.44,3.26,0.18,2.04,3.27,1305.68,0.18,0.63,166.5
2009-01-01T01:20:00.000Z,996.5,-7.62,265.81,-8.36,94.4,3.44,3.25,0.19,2.03,3.26,1305.69,0.19,0.5,118.6
2009-01-01T01:30:00.000Z,996.5,-7.91,265.52,-8.73,93.8,3.36,3.15,0.21,1.97,3.16,1307.17,0.28,0.75,188.5
2009-01-01T01:40:00.000Z,996.53,-8.43,264.99,-9.34,93.1,3.23,3.0,0.22,1.88,3.02,1309.85,0.59,0.88,185.0


## Transformación de datos

Se crean nuevas variables para enriquecer el análisis:

- año, mes, día, hora y trimestre;
- estación del año;
- franja horaria;
- diferencia entre temperatura y punto de rocío;
- clasificación de temperatura;
- clasificación de humedad;
- clasificación del viento;
- indicador de alta humedad;
- indicador de viento fuerte;
- indicador de posible condición extrema.

Estas variables permiten realizar análisis temporales y categóricos más claros.

In [0]:
# Transformaciones adicionales.

df_transformado = (
    df_limpio
    # Extraer componentes de fecha y crear nuevas columnas.
    .withColumn("anio", F.year("fecha_hora"))
    .withColumn("mes", F.month("fecha_hora"))
    .withColumn("dia", F.dayofmonth("fecha_hora"))
    .withColumn("hora", F.hour("fecha_hora"))
    .withColumn("trimestre", F.quarter("fecha_hora"))
    .withColumn("dia_semana_num", F.dayofweek("fecha_hora"))
    .withColumn(
        "dia_semana",
        F.when(F.col("dia_semana_num") == 1, "domingo")
         .when(F.col("dia_semana_num") == 2, "lunes")
         .when(F.col("dia_semana_num") == 3, "martes")
         .when(F.col("dia_semana_num") == 4, "miércoles")
         .when(F.col("dia_semana_num") == 5, "jueves")
         .when(F.col("dia_semana_num") == 6, "viernes")
         .otherwise("sábado")
    )
    # Clasificar estación del año basada en el mes.
    .withColumn(
        "estacion_anio",
        F.when(F.col("mes").isin(12, 1, 2), "invierno")
         .when(F.col("mes").isin(3, 4, 5), "primavera")
         .when(F.col("mes").isin(6, 7, 8), "verano")
         .otherwise("otoño")
    )
    # Clasificar franja horaria basada en la hora del día.
    .withColumn(
        "franja_horaria",
        F.when((F.col("hora") >= 0) & (F.col("hora") < 6), "madrugada")
         .when((F.col("hora") >= 6) & (F.col("hora") < 12), "mañana")
         .when((F.col("hora") >= 12) & (F.col("hora") < 18), "tarde")
         .otherwise("noche")
    )
    # Crear columna de diferencia entre temperatura y punto de rocío.
    .withColumn("diferencia_temp_rocio", F.round(F.col("temperatura_c") - F.col("punto_rocio_c"), 2))
    # Clasificar rango de temperatura en categorías.
    .withColumn(
        "rango_temperatura",
        F.when(F.col("temperatura_c") < 0, "frio_extremo")
         .when(F.col("temperatura_c") < 10, "frio")
         .when(F.col("temperatura_c") < 20, "templado")
         .when(F.col("temperatura_c") < 30, "calido")
         .otherwise("muy_calido")
    )
    # Clasificar nivel de humedad en categorías.
    .withColumn(
        "nivel_humedad",
        F.when(F.col("humedad_relativa_pct") < 40, "baja")
         .when(F.col("humedad_relativa_pct") < 70, "media")
         .otherwise("alta")
    )
    # Clasificar nivel de viento en categorías.
    .withColumn(
        "nivel_viento",
        F.when(F.col("velocidad_viento_ms") < 0.5, "calma")
         .when(F.col("velocidad_viento_ms") < 5, "brisa")
         .when(F.col("velocidad_viento_ms") < 10, "moderado")
         .otherwise("fuerte")
    )
    # Crear indicadores de alta humedad y viento fuerte, y una columna que marca condiciones extremas basadas en múltiples factores.
    .withColumn("alta_humedad", F.col("humedad_relativa_pct") >= 85)
    .withColumn("viento_fuerte", F.col("velocidad_viento_ms") >= 10)
    .withColumn(
        "condicion_extrema",
        (F.col("temperatura_c") <= -15) |
        (F.col("temperatura_c") >= 35) |
        (F.col("velocidad_viento_ms") >= 15) |
        (F.col("humedad_relativa_pct") >= 95)
    )
)

# Mostrar el número de columnas después de las transformaciones y las primeras filas para verificar el resultado.
print("Columnas después de transformar:", len(df_transformado.columns))
display(df_transformado.limit(10))

Columnas después de transformar: 31


fecha_hora,presion_mbar,temperatura_c,temperatura_potencial_k,punto_rocio_c,humedad_relativa_pct,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_g_kg,concentracion_agua_mmol_mol,densidad_aire_g_m3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados,anio,mes,dia,hora,trimestre,dia_semana_num,dia_semana,estacion_anio,franja_horaria,diferencia_temp_rocio,rango_temperatura,nivel_humedad,nivel_viento,alta_humedad,viento_fuerte,condicion_extrema
2009-01-01T00:10:00.000Z,996.52,-8.02,265.4,-8.9,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3,2009,1,1,0,1,5,jueves,invierno,madrugada,0.88,frio_extremo,alta,brisa,true,false,false
2009-01-01T00:20:00.000Z,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.8,0.72,1.5,136.1,2009,1,1,0,1,5,jueves,invierno,madrugada,0.87,frio_extremo,alta,brisa,true,false,false
2009-01-01T00:30:00.000Z,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.2,1.88,3.02,1310.24,0.19,0.63,171.6,2009,1,1,0,1,5,jueves,invierno,madrugada,0.8,frio_extremo,alta,calma,true,false,false
2009-01-01T00:40:00.000Z,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.5,198.0,2009,1,1,0,1,5,jueves,invierno,madrugada,0.76,frio_extremo,alta,calma,true,false,false
2009-01-01T00:50:00.000Z,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.0,0.32,0.63,214.3,2009,1,1,0,1,5,jueves,invierno,madrugada,0.77,frio_extremo,alta,calma,true,false,false
2009-01-01T01:00:00.000Z,996.5,-8.05,265.38,-8.78,94.4,3.33,3.14,0.19,1.96,3.15,1307.86,0.21,0.63,192.7,2009,1,1,1,1,5,jueves,invierno,madrugada,0.73,frio_extremo,alta,calma,true,false,false
2009-01-01T01:10:00.000Z,996.5,-7.62,265.81,-8.3,94.8,3.44,3.26,0.18,2.04,3.27,1305.68,0.18,0.63,166.5,2009,1,1,1,1,5,jueves,invierno,madrugada,0.68,frio_extremo,alta,calma,true,false,false
2009-01-01T01:20:00.000Z,996.5,-7.62,265.81,-8.36,94.4,3.44,3.25,0.19,2.03,3.26,1305.69,0.19,0.5,118.6,2009,1,1,1,1,5,jueves,invierno,madrugada,0.74,frio_extremo,alta,calma,true,false,false
2009-01-01T01:30:00.000Z,996.5,-7.91,265.52,-8.73,93.8,3.36,3.15,0.21,1.97,3.16,1307.17,0.28,0.75,188.5,2009,1,1,1,1,5,jueves,invierno,madrugada,0.82,frio_extremo,alta,calma,true,false,false
2009-01-01T01:40:00.000Z,996.53,-8.43,264.99,-9.34,93.1,3.23,3.0,0.22,1.88,3.02,1309.85,0.59,0.88,185.0,2009,1,1,1,1,5,jueves,invierno,madrugada,0.91,frio_extremo,alta,brisa,true,false,false


## Análisis con Spark DataFrames

A continuación se realizan diferentes análisis usando operaciones de Spark DataFrames.

Se incluyen tendencias temporales, rankings, agrupaciones, medias, máximos, mínimos y comparativas entre grupos.

#### Evolución mensual de temperatura, humedad, presión y viento

In [0]:
# Análisis mensual: temperatura media, humedad media, presión media, velocidad de viento media y número de mediciones por mes.
df_mensual = (
    df_transformado
    # Agrupar por año y mes, y calcular las métricas redondeando a 2 decimales para mejor presentación.
    .groupBy("anio", "mes")
    .agg(
        F.round(F.avg("temperatura_c"), 2).alias("temperatura_media_c"),
        F.round(F.avg("humedad_relativa_pct"), 2).alias("humedad_media_pct"),
        F.round(F.avg("presion_mbar"), 2).alias("presion_media_mbar"),
        F.round(F.avg("velocidad_viento_ms"), 2).alias("viento_medio_ms"),
        F.count("*").alias("num_mediciones")
    )
    .orderBy("anio", "mes")
)

# Mostrar el análisis mensual con las métricas calculadas.
display(df_mensual)

anio,mes,temperatura_media_c,humedad_media_pct,presion_media_mbar,viento_medio_ms,num_mediciones
2009,1,-3.63,84.99,988.99,1.78,4463
2009,2,0.17,84.72,985.63,2.05,4032
2009,3,3.99,78.47,986.12,2.43,4464
2009,4,11.89,69.18,987.52,1.85,4320
2009,5,13.43,70.97,992.11,2.08,4464
2009,6,14.25,73.89,988.85,2.11,4320
2009,7,17.99,71.42,987.79,2.13,4464
2009,8,18.88,65.49,990.85,1.7,4464
2009,9,14.29,77.4,993.37,1.77,4320
2009,10,7.56,84.17,990.03,1.99,4462


Este análisis sirve para ver cómo cambian las variables meteorológicas a lo largo del tiempo. La temperatura sigue un patrón muy claro: baja en invierno y sube en verano. La humedad suele comportarse al revés, siendo más alta en meses fríos y más baja en meses cálidos. La presión y el viento cambian menos, aunque también presentan algunos meses con valores más extremos. Además, al revisar el número de mediciones se detecta que enero de 2017 no es comparable con el resto de meses porque solo tiene un registro.

#### Temperaturas máxima y mínima por año

In [0]:
# Análisis anual: temperatura media, mínima y máxima, y humedad media por año.
df_temperaturas_anuales = (
    df_transformado
    # Agrupar por año y calcular las métricas redondeando a 2 decimales para mejor presentación.
    .groupBy("anio")
    .agg(
        F.round(F.avg("temperatura_c"), 2).alias("temperatura_media_c"),
        F.round(F.min("temperatura_c"), 2).alias("temperatura_minima_c"),
        F.round(F.max("temperatura_c"), 2).alias("temperatura_maxima_c"),
        F.round(F.avg("humedad_relativa_pct"), 2).alias("humedad_media_pct")
    )
    .orderBy("anio")
)

# Mostrar el análisis anual con las métricas calculadas.
display(df_temperaturas_anuales)

anio,temperatura_media_c,temperatura_minima_c,temperatura_maxima_c,humedad_media_pct
2009,8.83,-23.01,32.98,77.24
2010,7.46,-17.87,34.92,77.05
2011,9.3,-14.68,33.11,74.92
2012,9.66,-21.04,35.86,74.31
2013,9.09,-13.51,35.48,77.38
2014,10.7,-10.02,33.74,77.77
2015,10.51,-6.48,37.28,73.77
2016,9.99,-13.93,34.35,75.81
2017,-4.82,-4.82,-4.82,75.7


El análisis anual muestra que las temperaturas medias se mantienen relativamente estables, aunque con diferencias entre años. Destacan 2014 y 2015 como años más cálidos, mientras que 2010 presenta la media más baja entre los años completos. La humedad relativa media varía menos que la temperatura, manteniéndose en valores altos durante todo el periodo.

Para evitar interpretaciones incorrectas, el año 2017 no debería utilizarse en comparativas anuales, ya que contiene únicamente una medición. Mantenerlo en la tabla permite detectar esta anomalía de cobertura temporal, pero no debe considerarse representativo del comportamiento climático anual.

#### Distribución de mediciones por estación del año

In [0]:
# Análisis por estación del año: número de mediciones, temperatura media, humedad media y velocidad de viento media por estación.
df_estaciones = (
    df_transformado
    # Agrupar por estación del año y calcular las métricas redondeando a 2 decimales para mejor presentación, ordenando por número de mediciones.
    .groupBy("estacion_anio")
    .agg(
        F.count("*").alias("num_mediciones"),
        F.round(F.avg("temperatura_c"), 2).alias("temperatura_media_c"),
        F.round(F.avg("humedad_relativa_pct"), 2).alias("humedad_media_pct"),
        F.round(F.avg("velocidad_viento_ms"), 2).alias("viento_medio_ms")
    )
    .orderBy(F.desc("num_mediciones"))
)

# Mostrar el análisis por estación del año con las métricas calculadas.
display(df_estaciones)

estacion_anio,num_mediciones,temperatura_media_c,humedad_media_pct,viento_medio_ms
primavera,105983,9.17,70.61,2.18
verano,105963,18.04,69.46,1.96
otoño,104290,9.63,81.34,2.0
invierno,103968,0.78,82.92,2.38


El análisis por estación confirma una estacionalidad clara: el verano concentra las temperaturas más altas y menor humedad, mientras que el invierno presenta las temperaturas más bajas, mayor humedad y algo más de viento medio.

#### Ranking de meses más fríos y más cálidos

In [0]:
# Identificar los meses más fríos y más cálidos basados en la temperatura media mensual.
print("Meses más fríos:")
display(
    df_mensual
    .orderBy(F.asc("temperatura_media_c"))
    .limit(10)
)

# Identificar los meses más cálidos basados en la temperatura media mensual.
print("Meses más cálidos:")
display(
    df_mensual
    .orderBy(F.desc("temperatura_media_c"))
    .limit(10)
)

Meses más fríos:


anio,mes,temperatura_media_c,humedad_media_pct,presion_media_mbar,viento_medio_ms,num_mediciones
2010,12,-5.31,88.29,984.91,2.07,4464
2010,1,-5.0,86.45,987.78,2.15,4464
2017,1,-4.82,75.7,999.82,1.23,1
2009,1,-3.63,84.99,988.99,1.78,4463
2012,2,-3.09,78.67,999.21,2.08,4176
2010,2,-0.94,79.48,978.36,2.6,4032
2013,3,-0.8,77.56,983.84,2.55,4464
2013,2,-0.72,87.16,988.4,2.13,4032
2009,12,-0.49,86.39,981.62,2.37,4464
2011,2,-0.22,76.73,991.77,2.5,4032


Meses más cálidos:


anio,mes,temperatura_media_c,humedad_media_pct,presion_media_mbar,viento_medio_ms,num_mediciones
2015,8,20.93,67.9,989.51,1.88,4464
2010,7,20.75,64.8,989.74,1.76,4464
2015,7,20.6,62.9,988.41,2.12,4444
2013,7,20.33,66.16,993.04,1.8,4464
2014,7,20.14,70.68,987.25,1.79,4463
2012,8,19.62,64.23,990.38,1.89,4464
2016,7,19.61,67.84,990.44,1.88,4464
2009,8,18.88,65.49,990.85,1.7,4464
2016,8,18.72,65.53,992.75,1.84,4464
2013,8,18.56,68.73,991.55,1.66,4464


En el ranking de meses más fríos aparecen principalmente meses de invierno, como diciembre, enero y febrero. Sin embargo, también aparece marzo de 2013 con una temperatura media negativa. Aunque pueda resultar llamativo, no implica necesariamente un error, sino que ese mes concreto pudo presentar condiciones especialmente frías. Este tipo de resultado muestra la utilidad del análisis, ya que permite detectar meses atípicos dentro de la serie temporal.

#### Análisis de condiciones extremas

In [0]:
df_condiciones_extremas = (
    df_transformado
    # Agrupar por año y mes, y calcular el número de condiciones extremas, el número total de mediciones y el porcentaje de condiciones extremas, ordenando por porcentaje descendente.
    .groupBy("anio", "mes")
    .agg(
        F.sum(F.when(F.col("condicion_extrema"), 1).otherwise(0)).alias("num_condiciones_extremas"),
        F.count("*").alias("num_mediciones"),
        F.round(
            F.sum(F.when(F.col("condicion_extrema"), 1).otherwise(0)) / F.count("*") * 100,
            2
        ).alias("porcentaje_condiciones_extremas")
    )
    .orderBy(F.desc("porcentaje_condiciones_extremas"))
)

# Mostrar el análisis de condiciones extremas por mes con las métricas calculadas.
display(df_condiciones_extremas.limit(20))

anio,mes,num_condiciones_extremas,num_mediciones,porcentaje_condiciones_extremas
2011,11,1674,4320,38.75
2014,11,1584,4320,36.67
2010,11,1352,4320,31.3
2013,11,1229,4320,28.45
2014,9,1105,4225,26.15
2015,10,1153,4464,25.83
2013,2,924,4032,22.92
2009,12,1013,4464,22.69
2016,11,972,4320,22.5
2014,1,952,4464,21.33


El análisis de condiciones extremas no cuenta eventos meteorológicos independientes, sino mediciones individuales clasificadas como extremas. Como el dataset registra datos cada 10 minutos, un episodio de frío, humedad elevada o viento fuerte mantenido durante varios días puede generar cientos o miles de registros extremos.

Por ejemplo, noviembre de 2011 presenta un 38,75 % de mediciones extremas. Esto no significa que casi el 40 % del mes haya tenido eventos meteorológicos excepcionales distintos, sino que una parte importante de las mediciones del mes cumplió alguno de los umbrales definidos. Bastan varios días consecutivos de frío o humedad elevada para alcanzar ese porcentaje.

#### Comparativa por franja horaria

In [0]:
df_franja = (
    df_transformado
    # Agrupar por franja horaria y calcular las métricas redondeando a 2 decimales para mejor presentación, ordenando por franja horaria.
    .groupBy("franja_horaria")
    .agg(
        F.round(F.avg("temperatura_c"), 2).alias("temperatura_media_c"),
        F.round(F.avg("humedad_relativa_pct"), 2).alias("humedad_media_pct"),
        F.round(F.avg("velocidad_viento_ms"), 2).alias("viento_medio_ms"),
        F.count("*").alias("num_mediciones")
    )
    .orderBy("franja_horaria")
)

# Mostrar el análisis por franja horaria con las métricas calculadas.
display(df_franja)

franja_horaria,temperatura_media_c,humedad_media_pct,viento_medio_ms,num_mediciones
madrugada,6.89,85.53,1.6,105048
mañana,8.81,78.33,2.2,105036
noche,9.67,75.48,1.9,105048
tarde,12.4,64.78,2.83,105072


El análisis por franja horaria muestra un patrón diario claro: las madrugadas son más frías y húmedas, mientras que las tardes son más cálidas, menos húmedas y con mayor velocidad media del viento.

#### Relación entre variables meteorológicas

Se calcula la correlación entre algunas variables numéricas.  
Esto permite detectar relaciones como la habitual asociación inversa entre temperatura y humedad relativa.

In [0]:
# Análisis de correlaciones entre variables clave: temperatura, humedad, presión, velocidad del viento y densidad del aire.
correlaciones = [
    ("temperatura_c", "humedad_relativa_pct"),
    ("temperatura_c", "presion_mbar"),
    ("temperatura_c", "densidad_aire_g_m3"),
    ("velocidad_viento_ms", "velocidad_viento_max_ms"),
    ("temperatura_c", "presion_vapor_actual_mbar")
]

# Calcular y mostrar las correlaciones entre las variables seleccionadas.
for col1, col2 in correlaciones:
    valor_corr = df_transformado.stat.corr(col1, col2)
    print(f"Correlación entre {col1} y {col2}: {valor_corr:.4f}")

Correlación entre temperatura_c y humedad_relativa_pct: -0.5720
Correlación entre temperatura_c y presion_mbar: -0.0453
Correlación entre temperatura_c y densidad_aire_g_m3: -0.9634
Correlación entre velocidad_viento_ms y velocidad_viento_max_ms: 0.9573
Correlación entre temperatura_c y presion_vapor_actual_mbar: 0.8680


El análisis muestra relaciones bastante coherentes: cuando sube la temperatura suele bajar la humedad relativa, el aire se vuelve menos denso y aumenta la presión de vapor. Además, la velocidad media y máxima del viento se comportan de forma muy parecida.

## Visualizaciones básicas en Databricks

Databricks permite usar `display()` para crear gráficos de forma visual desde la interfaz.  
Las siguientes tablas están preparadas para generar gráficos de líneas, barras o áreas desde la propia celda.

In [0]:
# Dataset preparado para gráfico de evolución temporal mensual.
display(df_mensual)

anio,mes,temperatura_media_c,humedad_media_pct,presion_media_mbar,viento_medio_ms,num_mediciones
2009,1,-3.63,84.99,988.99,1.78,4463
2009,2,0.17,84.72,985.63,2.05,4032
2009,3,3.99,78.47,986.12,2.43,4464
2009,4,11.89,69.18,987.52,1.85,4320
2009,5,13.43,70.97,992.11,2.08,4464
2009,6,14.25,73.89,988.85,2.11,4320
2009,7,17.99,71.42,987.79,2.13,4464
2009,8,18.88,65.49,990.85,1.7,4464
2009,9,14.29,77.4,993.37,1.77,4320
2009,10,7.56,84.17,990.03,1.99,4462


In [0]:
# Dataset preparado para gráfico de temperatura media por estación del año.
display(df_estaciones)

estacion_anio,num_mediciones,temperatura_media_c,humedad_media_pct,viento_medio_ms
primavera,105983,9.17,70.61,2.18
verano,105963,18.04,69.46,1.96
otoño,104290,9.63,81.34,2.0
invierno,103968,0.78,82.92,2.38


In [0]:
# Dataset preparado para gráfico de condiciones extremas.
display(df_condiciones_extremas.orderBy("anio", "mes"))

anio,mes,num_condiciones_extremas,num_mediciones,porcentaje_condiciones_extremas
2009,1,874,4463,19.58
2009,2,634,4032,15.72
2009,3,522,4464,11.69
2009,4,306,4320,7.08
2009,5,228,4464,5.11
2009,6,364,4320,8.43
2009,7,227,4464,5.09
2009,8,125,4464,2.8
2009,9,611,4320,14.14
2009,10,795,4462,17.82


## Spark SQL

Se registra el DataFrame transformado como vista temporal para realizar consultas SQL.  
La práctica solicita un mínimo de tres consultas SQL relevantes; aquí se incluyen varias consultas orientadas al análisis climático.

In [0]:
# Crear vista temporal para consultas SQL.
df_transformado.createOrReplaceTempView("clima_jena")
print("Vista temporal creada: clima_jena")

Vista temporal creada: clima_jena


#### Consulta SQL: evolución anual de temperatura

In [0]:
sql_temperatura_anual = spark.sql("""
SELECT
    anio,
    ROUND(AVG(temperatura_c), 2) AS temperatura_media_c,
    ROUND(MIN(temperatura_c), 2) AS temperatura_minima_c,
    ROUND(MAX(temperatura_c), 2) AS temperatura_maxima_c,
    COUNT(*) AS num_mediciones
FROM clima_jena
GROUP BY anio
ORDER BY anio
""")

display(sql_temperatura_anual)

anio,temperatura_media_c,temperatura_minima_c,temperatura_maxima_c,num_mediciones
2009,8.83,-23.01,32.98,52557
2010,7.46,-17.87,34.92,52560
2011,9.3,-14.68,33.11,52560
2012,9.66,-21.04,35.86,52704
2013,9.09,-13.51,35.48,52559
2014,10.7,-10.02,33.74,52464
2015,10.51,-6.48,37.28,52540
2016,9.99,-13.93,34.35,52259
2017,-4.82,-4.82,-4.82,1


#### Consulta SQL: meses con más condiciones extremas

In [0]:
sql_condiciones_extremas = spark.sql("""
SELECT
    anio,
    mes,
    SUM(CASE WHEN condicion_extrema THEN 1 ELSE 0 END) AS num_condiciones_extremas,
    COUNT(*) AS num_mediciones,
    ROUND(SUM(CASE WHEN condicion_extrema THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_condiciones_extremas
FROM clima_jena
GROUP BY anio, mes
ORDER BY pct_condiciones_extremas DESC
LIMIT 15
""")

display(sql_condiciones_extremas)

anio,mes,num_condiciones_extremas,num_mediciones,pct_condiciones_extremas
2011,11,1674,4320,38.75
2014,11,1584,4320,36.67
2010,11,1352,4320,31.3
2013,11,1229,4320,28.45
2014,9,1105,4225,26.15
2015,10,1153,4464,25.83
2013,2,924,4032,22.92
2009,12,1013,4464,22.69
2016,11,972,4320,22.5
2014,1,952,4464,21.33


#### Consulta SQL: comportamiento por estación del año

In [0]:
sql_estaciones = spark.sql("""
SELECT
    estacion_anio,
    ROUND(AVG(temperatura_c), 2) AS temperatura_media_c,
    ROUND(AVG(humedad_relativa_pct), 2) AS humedad_media_pct,
    ROUND(AVG(velocidad_viento_ms), 2) AS viento_medio_ms,
    ROUND(AVG(presion_mbar), 2) AS presion_media_mbar,
    COUNT(*) AS num_mediciones
FROM clima_jena
GROUP BY estacion_anio
ORDER BY temperatura_media_c DESC
""")

display(sql_estaciones)

estacion_anio,temperatura_media_c,humedad_media_pct,viento_medio_ms,presion_media_mbar,num_mediciones
verano,18.04,69.46,1.96,989.25,105963
otoño,9.63,81.34,2.0,989.74,104290
primavera,9.17,70.61,2.18,989.17,105983
invierno,0.78,82.92,2.38,988.69,103968


#### Consulta SQL: franjas horarias con más humedad alta

In [0]:
sql_humedad_franja = spark.sql("""
SELECT
    franja_horaria,
    COUNT(*) AS num_mediciones,
    SUM(CASE WHEN alta_humedad THEN 1 ELSE 0 END) AS mediciones_alta_humedad,
    ROUND(SUM(CASE WHEN alta_humedad THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_alta_humedad
FROM clima_jena
GROUP BY franja_horaria
ORDER BY pct_alta_humedad DESC
""")

display(sql_humedad_franja)

franja_horaria,num_mediciones,mediciones_alta_humedad,pct_alta_humedad
madrugada,105048,61261,58.32
mañana,105036,40765,38.81
noche,105048,34903,33.23
tarde,105072,16135,15.36


#### Consulta SQL: Días con mayor Amplitud Térmica (El mayor "salto" de temperatura en 24h)

In [0]:
sql_amplitud_termica = spark.sql("""
SELECT
    anio, 
    mes, 
    dia,
    ROUND(MAX(temperatura_c), 2) AS temp_maxima,
    ROUND(MIN(temperatura_c), 2) AS temp_minima,
    ROUND(MAX(temperatura_c) - MIN(temperatura_c), 2) AS amplitud_termica
FROM clima_jena
GROUP BY anio, mes, dia
ORDER BY amplitud_termica DESC
LIMIT 10
""")

display(sql_amplitud_termica)

anio,mes,dia,temp_maxima,temp_minima,amplitud_termica
2012,3,16,22.1,0.38,21.72
2015,6,5,31.75,10.05,21.7
2011,5,26,26.8,5.59,21.21
2016,5,22,30.66,9.64,21.02
2014,3,9,19.33,-1.42,20.75
2015,7,7,33.81,13.4,20.41
2012,8,19,35.65,15.25,20.4
2015,8,6,34.41,14.1,20.31
2009,9,1,30.11,9.89,20.22
2009,5,17,25.92,5.99,19.93


#### Consulta SQL: Rosa de los Vientos simplificada (Análisis de sectores)

Esta consulta sirve para entender mejor de dónde suele venir el viento. En vez de mirar el valor exacto en grados, se agrupa en Norte, Este, Sur y Oeste. Así es más fácil ver qué dirección aparece más veces y si el viento de alguna zona suele ser más fuerte que el resto.

In [0]:
sql_rosa_vientos = spark.sql("""
SELECT
    CASE
        WHEN direccion_viento_grados >= 315 OR direccion_viento_grados < 45 THEN 'Norte'
        WHEN direccion_viento_grados >= 45 AND direccion_viento_grados < 135 THEN 'Este'
        WHEN direccion_viento_grados >= 135 AND direccion_viento_grados < 225 THEN 'Sur'
        WHEN direccion_viento_grados >= 225 AND direccion_viento_grados < 315 THEN 'Oeste'
    END AS sector_viento,
    COUNT(*) as frecuencia_mediciones,
    ROUND(AVG(velocidad_viento_ms), 2) AS velocidad_media_ms,
    ROUND(MAX(velocidad_viento_max_ms), 2) AS racha_maxima_ms
FROM clima_jena
GROUP BY sector_viento
ORDER BY frecuencia_mediciones DESC
""")

display(sql_rosa_vientos)

sector_viento,frecuencia_mediciones,velocidad_media_ms,racha_maxima_ms
Sur,186750,2.26,23.5
Oeste,109397,1.97,20.23
Norte,79463,2.22,14.38
Este,44594,1.82,14.57


#### Consulta SQL: Recuento de "Días de Helada" por año

In [0]:
sql_dias_helada = spark.sql("""
WITH TemperaturasDiarias AS (
    SELECT 
        anio, 
        mes, 
        dia, 
        MAX(temperatura_c) AS temp_max_diaria
    FROM clima_jena
    GROUP BY anio, mes, dia
)
SELECT 
    anio, 
    COUNT(*) AS dias_con_maxima_bajo_cero
FROM TemperaturasDiarias
WHERE temp_max_diaria <= 0
GROUP BY anio
ORDER BY anio
""")

display(sql_dias_helada)

anio,dias_con_maxima_bajo_cero
2009,31
2010,68
2011,17
2012,23
2013,34
2014,12
2015,5
2016,8
2017,1


## Detección sencilla de anomalías mediante percentiles

Como enriquecimiento de la práctica, se realiza una detección simple de anomalías usando percentiles.

Se consideran valores atípicos aquellos que se encuentran por debajo del percentil 1 o por encima del percentil 99 en variables seleccionadas.

In [0]:
variables_anomalias = [
    "temperatura_c",
    "presion_mbar",
    "humedad_relativa_pct",
    "velocidad_viento_ms"
]

percentiles = {}
for c in variables_anomalias:
    p01, p99 = df_transformado.approxQuantile(c, [0.01, 0.99], 0.01)
    percentiles[c] = (p01, p99)
    print(f"{c}: P01={p01}, P99={p99}")

df_anomalias = df_transformado
for c, (p01, p99) in percentiles.items():
    df_anomalias = df_anomalias.withColumn(
        f"anomalia_{c}",
        (F.col(c) < F.lit(p01)) | (F.col(c) > F.lit(p99))
    )

df_anomalias = df_anomalias.withColumn(
    "anomalia_percentil",
    F.col("anomalia_temperatura_c") |
    F.col("anomalia_presion_mbar") |
    F.col("anomalia_humedad_relativa_pct") |
    F.col("anomalia_velocidad_viento_ms")
)

print("Número de registros con posible anomalía por percentiles:", df_anomalias.filter(F.col("anomalia_percentil")).count())
display(df_anomalias.filter(F.col("anomalia_percentil")).limit(50))

temperatura_c: P01=-23.01, P99=37.28
presion_mbar: P01=913.6, P99=1015.35
humedad_relativa_pct: P01=12.95, P99=100.0
velocidad_viento_ms: P01=0.0, P99=14.63
Número de registros con posible anomalía por percentiles: 0


fecha_hora,presion_mbar,temperatura_c,temperatura_potencial_k,punto_rocio_c,humedad_relativa_pct,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_g_kg,concentracion_agua_mmol_mol,densidad_aire_g_m3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados,anio,mes,dia,hora,trimestre,dia_semana_num,dia_semana,estacion_anio,franja_horaria,diferencia_temp_rocio,rango_temperatura,nivel_humedad,nivel_viento,alta_humedad,viento_fuerte,condicion_extrema,anomalia_temperatura_c,anomalia_presion_mbar,anomalia_humedad_relativa_pct,anomalia_velocidad_viento_ms,anomalia_percentil


## Exportación final en Parquet

Se guarda el dataset final limpio y transformado en formato **Parquet**.

Se elige Parquet porque:

- es un formato columnar eficiente;
- comprime bien los datos;
- conserva los tipos de datos;
- es adecuado para análisis con Spark;
- es compatible con otros servicios Big Data.


In [0]:
(
    df_transformado
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH_PARQUET)
)

print("Dataset limpio exportado correctamente en Parquet:")
print(OUTPUT_PATH_PARQUET)

Dataset limpio exportado correctamente en Parquet:
/Volumes/workspace/default/dataset/jena_climate_limpio.parquet


In [0]:
# Comprobación de lectura del Parquet exportado
df_parquet = spark.read.parquet(OUTPUT_PATH_PARQUET)

print("Registros leídos desde Parquet:", df_parquet.count())
print("Columnas leídas desde Parquet:", len(df_parquet.columns))

display(df_parquet.limit(10))

Registros leídos desde Parquet: 420204
Columnas leídas desde Parquet: 31


fecha_hora,presion_mbar,temperatura_c,temperatura_potencial_k,punto_rocio_c,humedad_relativa_pct,presion_vapor_max_mbar,presion_vapor_actual_mbar,deficit_presion_vapor_mbar,humedad_especifica_g_kg,concentracion_agua_mmol_mol,densidad_aire_g_m3,velocidad_viento_ms,velocidad_viento_max_ms,direccion_viento_grados,anio,mes,dia,hora,trimestre,dia_semana_num,dia_semana,estacion_anio,franja_horaria,diferencia_temp_rocio,rango_temperatura,nivel_humedad,nivel_viento,alta_humedad,viento_fuerte,condicion_extrema
2015-08-06T23:10:00.000Z,989.47,25.41,299.48,13.22,46.81,32.51,15.22,17.29,9.62,15.38,1147.76,0.62,1.04,61.71,2015,8,6,23,3,5,jueves,verano,noche,12.19,calido,media,brisa,false,false,false
2015-08-06T23:20:00.000Z,989.45,24.95,299.02,12.92,47.17,31.63,14.92,16.71,9.43,15.08,1149.64,0.51,1.04,357.7,2015,8,6,23,3,5,jueves,verano,noche,12.03,calido,media,brisa,false,false,false
2015-08-06T23:30:00.000Z,989.44,24.74,298.81,13.15,48.48,31.24,15.14,16.09,9.58,15.31,1150.34,0.42,0.62,93.0,2015,8,6,23,3,5,jueves,verano,noche,11.59,calido,media,calma,false,false,false
2015-08-06T23:40:00.000Z,989.38,24.41,298.48,13.05,49.12,30.63,15.05,15.58,9.51,15.21,1151.59,0.48,0.96,30.74,2015,8,6,23,3,5,jueves,verano,noche,11.36,calido,media,calma,false,false,false
2015-08-06T23:50:00.000Z,989.35,24.2,298.27,12.92,49.31,30.25,14.91,15.33,9.43,15.08,1152.43,0.44,0.8,83.0,2015,8,6,23,3,5,jueves,verano,noche,11.28,calido,media,calma,false,false,false
2015-08-07T00:00:00.000Z,989.35,23.94,298.01,13.27,51.25,29.78,15.26,14.52,9.65,15.43,1153.28,0.47,0.76,45.59,2015,8,7,0,3,6,viernes,verano,madrugada,10.67,calido,media,calma,false,false,false
2015-08-07T00:10:00.000Z,989.36,23.65,297.72,12.81,50.63,29.26,14.82,14.45,9.37,14.98,1154.62,0.51,1.56,246.9,2015,8,7,0,3,6,viernes,verano,madrugada,10.84,calido,media,brisa,false,false,false
2015-08-07T00:20:00.000Z,989.35,23.46,297.53,12.36,49.7,28.93,14.38,14.55,9.09,14.53,1155.54,2.13,2.88,213.7,2015,8,7,0,3,6,viernes,verano,madrugada,11.1,calido,media,brisa,false,false,false
2015-08-07T00:30:00.000Z,989.36,22.44,296.51,12.45,53.19,27.2,14.47,12.73,9.15,14.62,1159.5,1.47,2.4,208.7,2015,8,7,0,3,6,viernes,verano,madrugada,9.99,calido,media,brisa,false,false,false
2015-08-07T00:40:00.000Z,989.31,22.34,296.41,12.71,54.44,27.04,14.72,12.32,9.31,14.88,1159.72,1.63,2.54,206.0,2015,8,7,0,3,6,viernes,verano,madrugada,9.63,calido,media,brisa,false,false,false


## Principales conclusiones

A partir del proceso realizado se pueden extraer las siguientes conclusiones:

1. La limpieza se ha centrado en corregir tipos de datos, eliminar duplicados y filtrar valores físicamente incoherentes.
2. La columna temporal ha sido clave para crear nuevas variables como año, mes, trimestre, estación del año y franja horaria.
3. Las variables derivadas permiten analizar el comportamiento climático desde una perspectiva temporal y categórica.
4. Spark DataFrames facilita las agregaciones sobre cientos de miles de registros de forma sencilla y eficiente.
5. Spark SQL permite consultar el dataset transformado con una sintaxis similar a SQL tradicional.
6. Parquet es un formato adecuado para guardar el resultado final por su eficiencia y compatibilidad con entornos Big Data.

## Resumen de limpieza y transformaciones aplicadas

**Problemas revisados:**

- valores nulos.
- duplicados completos.
- fechas duplicadas.
- tipos de datos incorrectos.
- humedad fuera de rango.
- viento negativo.
- dirección del viento incorrecta.
- temperaturas extremas.
- presión atmosférica incoherente.

**Transformaciones aplicadas:**

- renombrado de columnas.
- conversión de fecha a timestamp.
- conversión de variables meteorológicas a double.
- creación de columnas temporales.
- clasificación por estación del año.
- clasificación por franja horaria.
- clasificación de temperatura, humedad y viento.
- creación de indicadores booleanos.
- detección simple de anomalías por reglas y percentiles.

**Análisis realizados:**

- evolución mensual de variables climáticas.
- temperaturas anuales máximas, mínimas y medias.
- comportamiento por estación del año.
- ranking de meses fríos y cálidos.
- análisis de condiciones extremas.
- comparativa por franja horaria.
- correlación entre variables.
- consultas SQL sobre la vista temporal.